# Validate and score a saved drive
Use the competition environment and **Run All**. Interactive validation needs the ego `solution.xml` and the matching **simulated** `scenario.xml` from the same run. A Solution XML alone contains no background traffic.

By default, this notebook generates a temporary Augsburg example with the supplied SMP A* planner. To check an existing export **without rerunning your planner**, set `pair_directory` to its folder in the first cell. The same validation and scoring cells work for both.


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
from commonroad.common.file_reader import CommonRoadFileReader
from commonroad.common.solution import CommonRoadSolutionReader
from planner.common.scenarios import ASSETS
from planner.runtime.artifacts import export_result
from planner.runtime.execution import run_spec
from planner.spec import smp_spec

root = Path.cwd()
if root.name == "tutorials":
    root = root.parent

pair_directory = None  # None generates the temporary example; set a path to read saved files.
# pair_directory = root / "solutions" / "DEU_AugsburgCRC26P-23_16300_I-1"

with TemporaryDirectory(prefix="commonroad-validation-example-") as directory:
    if pair_directory is None:
        scenario_path = ASSETS / "scenarios/public/DEU_AugsburgCRC26P-23_16300_I-1.xml"
        result = run_spec(scenario_path, smp_spec())
        pair_directory = export_result(result, directory)
    scenario, problems = CommonRoadFileReader(str(pair_directory / "scenario.xml")).open()
    solution = CommonRoadSolutionReader.open(str(pair_directory / "solution.xml"))


## Check validity
The runner and this notebook use `validate_solution()`, a reporting helper around the unchanged upstream CommonRoad `valid_solution()` used by the search bonus exercise. The checker covers planning problems, initial states, goals, obstacle/ego collisions, road boundaries and vehicle dynamics, with the same physical limits and tolerances.

The result has named fields: `valid`, `feasible` and `message`. `feasible=False` means dynamics did not pass. `feasible=None` means the checker stopped without returning feasibility results, for example after finding a collision or a missed goal. Inspect `message` for the reason. Unexpected errors still raise normally.

For reconstructed controls and trajectories, call `simulation.solution_check.valid_solution()` directly. It re-exports the upstream function and returns `(valid, feasibility_details)`, where each planning-problem entry contains `(feasible, inputs, trajectory)`. It can raise `SolutionCheckerException`; the first return value is overall validity.


In [ ]:
from dataclasses import asdict
from planner.common.validation import validate_solution

validation = validate_solution(scenario, problems, solution)
asdict(validation)


## Compute SM1
Only valid drives receive a score. Lower SM1 is better within the same scenario.

| Cost term | Weight |
| --- | ---: |
| Acceleration | 50 |
| Steering angle | 50 |
| Steering rate | 50 |
| Path length | 1 |
| Speed deviation | 20 |
| Lane-heading deviation | 50 |

This is the final evaluation cost. A planner can use a different internal objective; SMP A* accumulates travel time.


In [ ]:
from planner.common.scoring import CostFunctionEvaluator

if validation.valid:
    evaluator = CostFunctionEvaluator.init_from_solution(solution)
    score = evaluator.evaluate_solution(scenario, problems, solution).total_costs
    print("SM1:", score)
else:
    print("Fix the invalid drive before comparing costs.")


These are local validity and scoring checks. See the [submission instructions](../README.md#save-results-for-submission) for the required files and source.
